# 03 · Visualização e análise exploratória

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

Antes de treinar qualquer modelo é preciso **olhar** os dados. Um gráfico bem escolhido mostra em segundos o que nenhuma tabela de médias revela: valores impossíveis, grupos com comportamento diferente, relações que não são lineares. Este notebook ensina a escolher o gráfico pela pergunta, a construí-lo com controle e a conduzir uma análise exploratória que termine em decisões para o modelo.

## O que você será capaz de fazer ao final

1. Construir figuras com a interface orientada a objetos do Matplotlib.
2. Escolher a forma do gráfico a partir da pergunta.
3. Representar distribuições, comparações, relações e evolução no tempo.
4. Mostrar a incerteza de uma comparação entre grupos.
5. Usar cor com função definida e reconhecer gráficos que enganam.
6. Usar seaborn e Plotly quando eles economizam trabalho.
7. Seguir um roteiro de análise exploratória voltado a um alvo.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("matplotlib", matplotlib.__version__, "· pandas", pd.__version__)

# Aparência padrão das figuras deste notebook: discreta, para que os dados apareçam
plt.rcParams.update({
    "figure.figsize": (6.5, 3.8), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 11,
})

AZUL, LARANJA, VERDE, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6b6b6b"     # paleta segura para daltonismo


def titulo_de(ax):
    """Título de um Axes, esteja ele à esquerda, no centro ou à direita (usada nas verificações)."""
    return ax.get_title(loc="left") or ax.get_title(loc="center") or ax.get_title(loc="right")

In [ ]:
URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

bruto = pd.read_csv(ARQUIVO, low_memory=False, dtype={"product_id": str}, parse_dates=["submission_date"])
idade = (2018 - bruto["reviewer_birth_year"]).where(lambda s: s.between(14, 100))

dados = bruto.dropna(subset=["recommend_to_a_friend"]).assign(          # mesma preparação do notebook 02
    nao_recomenda=lambda t: (t["recommend_to_a_friend"] == "No").astype(int),
    idade=idade,
    faixa_etaria=pd.cut(idade, bins=[13, 24, 39, 59, 100], labels=["14-24", "25-39", "40-59", "60+"]),
    texto=lambda t: t["review_title"].fillna("") + ". " + t["review_text"].fillna(""),
    palavras=lambda t: t["texto"].str.split().str.len(),
    hora=lambda t: t["submission_date"].dt.hour,
    dia=lambda t: t["submission_date"].dt.normalize(),
)
TAXA_GERAL = dados["nao_recomenda"].mean()
print(dados.shape, f"· taxa geral de não recomendação: {TAXA_GERAL:.1%}")

## 1. Por que olhar

Os quatro conjuntos abaixo, criados pelo estatístico Francis Anscombe, têm praticamente as mesmas médias, as mesmas variâncias e a mesma correlação.

In [ ]:
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
quarteto = {
    "I":   (x123, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  (x123, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (x123, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  ([8] * 7 + [19] + [8] * 3, [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}
pd.DataFrame({
    nome: {"média de x": np.mean(x), "média de y": np.mean(y), "variância de y": np.var(y, ddof=1), "correlação": np.corrcoef(x, y)[0, 1]}
    for nome, (x, y) in quarteto.items()
}).round(2)

In [ ]:
fig, eixos = plt.subplots(1, 4, figsize=(11, 2.8), sharex=True, sharey=True)
for eixo, (nome, (x, y)) in zip(eixos, quarteto.items()):
    eixo.scatter(x, y, color=AZUL, s=28)
    eixo.set_title(f"Conjunto {nome}")
fig.tight_layout()
plt.show()

Uma relação linear com ruído, uma curva, uma reta com um ponto discrepante e um conjunto em que um único ponto cria toda a correlação. Os resumos numéricos são idênticos; as histórias, completamente diferentes. **Resumos escondem; gráficos revelam.**

## 2. Anatomia de uma figura do Matplotlib

O Matplotlib tem dois objetos centrais:

- a **`Figure`** é a tela inteira;
- cada **`Axes`** é um gráfico dentro dela, com os seus eixos, o seu título e os seus dados.

```
 ┌──────────────── Figure ────────────────┐
 │  ┌──── Axes ────┐   ┌──── Axes ────┐   │
 │  │ título       │   │              │   │
 │  │ ▂▅▇▅▂ dados  │   │   ╱╲╱        │   │
 │  │ rótulos      │   │              │   │
 │  └──────────────┘   └──────────────┘   │
 └────────────────────────────────────────┘
```

Há duas formas de usar a biblioteca. A interface `plt.algo(...)` age sobre "o gráfico atual" e serve para um gráfico rápido. A interface **orientada a objetos**, em que você cria os objetos e chama métodos deles, é a que dá controle e a que usaremos.

In [ ]:
fig, ax = plt.subplots()                                   # cria a figura e um Axes

ax.hist(dados["idade"].dropna(), bins=30, color=AZUL)      # desenha no Axes
ax.set_xlabel("Idade do cliente (anos)")
ax.set_ylabel("Avaliações")
ax.set_title("A maior parte dos clientes tem entre 25 e 45 anos")

fig.savefig("idades.png", dpi=150, bbox_inches="tight")    # salva em arquivo
plt.show()

Três hábitos que já aparecem aqui:

1. **Todo eixo tem rótulo com unidade.** "Idade do cliente (anos)", e não `idade`.
2. **O título diz a conclusão**, e não o nome das variáveis. "Distribuição de idade" obriga o leitor a tirar a conclusão sozinho.
3. **Moldura discreta:** sem bordas supérfluas, grade clara. Isso foi configurado uma vez, em `plt.rcParams`, na primeira célula.

`plt.subplots(linhas, colunas)` cria vários `Axes` de uma vez e os devolve em um *array*.

### ✏️ Exercício 1 (★): seu primeiro `Axes`

Crie uma figura com um `Axes` chamado `ax1` contendo o histograma do número de palavras (`dados["palavras"]`) das avaliações com **até 100 palavras**, com exatamente **40 intervalos**. Coloque rótulos nos dois eixos e um título.

In [ ]:
# ✏️ Exercício 1
# SEU CÓDIGO AQUI
fig, ax1 = plt.subplots()
...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
assert len(ax1.patches) == 40, "o histograma deve ter 40 intervalos (bins=40)"
assert ax1.get_xlabel() and ax1.get_ylabel() and titulo_de(ax1), "faltam rótulos nos eixos ou o título"
assert ax1.patches[-1].get_x() + ax1.patches[-1].get_width() <= 100.5, "use apenas as avaliações com até 100 palavras"
print("Exercício 1 correto.")

## 3. A pergunta escolhe o gráfico

Antes de pensar em cores ou em bibliotecas, formule a pergunta. Ela decide a forma.

| A pergunta é sobre... | Exemplo | Forma indicada |
|---|---|---|
| **distribuição** de uma variável | "que idades têm os clientes?" | histograma; diagrama de caixa para comparar grupos |
| **comparação** entre categorias | "qual categoria tem mais reclamações?" | barras **ordenadas**; pontos com barras de erro |
| **relação** entre duas variáveis | "textos longos têm notas menores?" | dispersão; diagrama de caixa; mapa de calor |
| **evolução** no tempo | "a insatisfação está aumentando?" | linha |
| **composição** de um todo | "como as notas se distribuem em cada categoria?" | barras empilhadas até 100% |
| **um único número** | "qual é a taxa geral?" | escreva o número; não faça gráfico |

Duas formas a evitar: **pizza** com mais de duas ou três fatias (o olho compara mal ângulos) e gráficos **3D** decorativos (distorcem tamanhos).

## 4. Distribuições

### O número de intervalos muda a história

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(11, 2.8), sharey=False)
for eixo, n in zip(eixos, [5, 30, 86]):
    eixo.hist(dados["idade"].dropna(), bins=n, color=AZUL)
    eixo.set_title(f"{n} intervalos")
    eixo.set_xlabel("Idade (anos)")
fig.tight_layout()
plt.show()

Com poucos intervalos a forma some; com um intervalo por ano de idade aparecem picos que são artefato de arredondamento ou de preenchimento do cadastro. Experimente sempre dois ou três valores.

### Distribuições assimétricas

O tamanho dos textos tem cauda longa: a maioria é curta e alguns são enormes. Na escala comum quase tudo fica espremido à esquerda. A escala **logarítmica** dá o mesmo espaço a cada multiplicação por dez.

In [ ]:
fig, (esq, dir_) = plt.subplots(1, 2, figsize=(10, 3))

esq.hist(dados["palavras"], bins=60, color=AZUL)
esq.set_title("Escala linear: a cauda esconde a forma")
esq.set_xlabel("Palavras")

intervalos = np.logspace(0, np.log10(dados["palavras"].max()), 40)        # intervalos de largura crescente
dir_.hist(dados["palavras"], bins=intervalos, color=AZUL)
dir_.set_xscale("log")
dir_.set_title("Escala logarítmica: a forma aparece")
dir_.set_xlabel("Palavras (escala logarítmica)")
fig.tight_layout()
plt.show()

### Comparar distribuições: o diagrama de caixa

Para comparar a distribuição de uma variável numérica entre grupos, vários histogramas sobrepostos ficam ilegíveis. O **diagrama de caixa** resume cada grupo em cinco números.

```
        ┌─────┬─────┐
 ├──────┤     │     ├──────────┤        ○  ○
        └─────┴─────┘
 mínimo   25%  mediana 75%    máximo*    valores discrepantes
                               * até 1,5 vez a altura da caixa
```

In [ ]:
grupos = [dados.loc[dados["overall_rating"] == nota, "palavras"] for nota in range(1, 6)]

fig, ax = plt.subplots()
ax.boxplot(grupos, tick_labels=["1", "2", "3", "4", "5"], showfliers=False,          # showfliers=False omite os discrepantes
           patch_artist=True, boxprops={"facecolor": "#cfe1f7", "edgecolor": AZUL}, medianprops={"color": AZUL, "linewidth": 2})
ax.set_xlabel("Nota da avaliação")
ax.set_ylabel("Palavras na avaliação")
ax.set_title("Quem dá nota baixa escreve mais")
plt.show()

print(dados.groupby("overall_rating")["palavras"].median().to_string())

A mediana cai de 27 palavras, na nota 1, para 16, nas notas 4 e 5. Esse é um primeiro achado útil para o modelo: **o tamanho do texto carrega sinal** sobre a satisfação.

### ✏️ Exercício 2 (★★): idade por nota

Faça, em um `Axes` chamado `ax2`, os diagramas de caixa da **idade** para cada nota de 1 a 5, sem os valores discrepantes, com rótulos e título. Lembre-se de retirar as idades ausentes de cada grupo.

In [ ]:
# ✏️ Exercício 2
# SEU CÓDIGO AQUI
fig, ax2 = plt.subplots()
...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
assert [t.get_text() for t in ax2.get_xticklabels()] == ["1", "2", "3", "4", "5"], "um diagrama de caixa por nota, com os rótulos 1 a 5"
assert ax2.get_xlabel() and ax2.get_ylabel() and titulo_de(ax2)
_medianas = [linha.get_ydata()[0] for linha in ax2.lines if len(linha.get_ydata()) == 2 and linha.get_ydata()[0] == linha.get_ydata()[1]]
assert any(30 < m < 45 for m in _medianas), "as medianas de idade deveriam ficar entre 30 e 45 anos: os ausentes foram retirados?"
print("Exercício 2 correto.")

**Pergunta escrita.** Compare o gráfico do exercício 2 com o do tamanho do texto. O que cada um sugere sobre a utilidade da variável para prever a satisfação?

✏️ **Sua resposta:**

_Escreva aqui._

## 5. Comparar grupos

### Barras: ordenadas e partindo do zero

Barras comparam **comprimentos**. Duas regras decorrem disso: o eixo de valor começa em **zero** (senão o comprimento mente) e as barras vêm **ordenadas pelo valor** (a menos que a categoria tenha ordem própria, como faixas etárias ou meses). Nomes longos pedem barras horizontais.

In [ ]:
por_categoria = (
    dados.groupby("site_category_lv1")
    .agg(n=("nao_recomenda", "size"), taxa=("nao_recomenda", "mean"))
    .query("n >= 3000")
    .sort_values("taxa")
)

fig, ax = plt.subplots(figsize=(7, 4.6))
barras = ax.barh(por_categoria.index, por_categoria["taxa"], color=AZUL)
ax.bar_label(barras, labels=[f"{v:.0%}" for v in por_categoria["taxa"]], padding=3, fontsize=8)      # valor ao lado da barra
ax.axvline(TAXA_GERAL, color=CINZA, linestyle="--", linewidth=1)
ax.text(TAXA_GERAL, len(por_categoria) - 0.3, f" taxa geral: {TAXA_GERAL:.0%}", color=CINZA, fontsize=8, va="bottom")
ax.set_xlabel("Clientes que não recomendam")
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Em móveis, 45% dos clientes não recomendam a compra")
ax.grid(axis="y", visible=False)
plt.show()

Repare na **linha de referência** com a taxa geral: ela transforma cada barra em uma comparação ("acima ou abaixo do normal?"). E repare que a cor é uma só: todas as barras medem a mesma coisa, e cores diferentes sugeririam diferenças que não existem.

### Toda comparação tem incerteza

Uma taxa calculada com 80 avaliações é muito menos confiável que uma calculada com 20 mil. Para uma proporção $p$ medida em $n$ casos, o **erro padrão** é

$$\text{EP} = \sqrt{\frac{p\,(1 - p)}{n}}$$

e o intervalo de $p \pm 1{,}96 \cdot \text{EP}$ contém o valor verdadeiro em cerca de 95% das vezes. Mostrar esse intervalo evita tirar conclusões de diferenças que são puro acaso.

In [ ]:
por_sub = (
    dados.groupby("site_category_lv2")
    .agg(n=("nao_recomenda", "size"), taxa=("nao_recomenda", "mean"))
    .query("n >= 60")
    .assign(margem=lambda t: 1.96 * np.sqrt(t["taxa"] * (1 - t["taxa"]) / t["n"]))
)
extremos = pd.concat([por_sub.nlargest(8, "taxa"), por_sub[por_sub["n"] >= 2000].nlargest(4, "taxa")]).drop_duplicates().sort_values("taxa")

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.errorbar(extremos["taxa"], extremos.index, xerr=extremos["margem"], fmt="o", color=AZUL, ecolor=CINZA, capsize=3, markersize=5)
for nome, linha in extremos.iterrows():
    ax.text(0.99, nome, f"n = {int(linha['n'])}", transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=8, color=CINZA)
ax.axvline(TAXA_GERAL, color=CINZA, linestyle="--", linewidth=1)
ax.set_xlabel("Clientes que não recomendam (intervalo de 95%)")
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Subcategorias pequenas têm taxas extremas e incertas")
ax.grid(axis="y", visible=False)
plt.show()

As subcategorias com as taxas mais altas são, quase todas, as que têm **poucas avaliações**: os seus intervalos são largos. Entre as subcategorias grandes, os intervalos são estreitos e as diferenças, reais. Esse padrão é geral: **os extremos de qualquer ranking tendem a ser ocupados pelos grupos pequenos**, por acaso. Quem ordena uma tabela e olha o topo sem conferir o tamanho dos grupos é enganado com frequência.

### ✏️ Exercício 3 (★★): taxa por estado, com incerteza

1. Escreva `taxa_por_grupo(tabela, coluna, minimo=1)`, que devolve uma tabela indexada pelos valores de `coluna`, com as colunas `n`, `taxa` (média de `nao_recomenda`) e `margem` (1,96 vez o erro padrão), apenas para grupos com `n >= minimo`, ordenada pela taxa em ordem crescente.
2. Em um `Axes` chamado `ax3`, desenhe a taxa de cada **estado** como um ponto com a sua barra de erro horizontal, com rótulo no eixo e título.

In [ ]:
# ✏️ Exercício 3
def taxa_por_grupo(tabela, coluna, minimo=1):
    # SEU CÓDIGO AQUI
    ...

# SEU CÓDIGO AQUI
por_estado = ...
fig, ax3 = plt.subplots(figsize=(6.5, 6))
...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
_t = taxa_por_grupo(dados, "reviewer_gender")
assert list(_t.columns) == ["n", "taxa", "margem"] and set(_t.index) == {"F", "M"}
assert np.allclose(_t["margem"], 1.96 * np.sqrt(_t["taxa"] * (1 - _t["taxa"]) / _t["n"])) and _t["taxa"].is_monotonic_increasing
assert len(taxa_por_grupo(dados, "site_category_lv2", minimo=2000)) < 20, "o argumento minimo deve filtrar os grupos pequenos"
assert len(ax3.containers) >= 1 and ax3.get_xlabel() and titulo_de(ax3), "use ax3.errorbar e coloque rótulo e título"
print("Exercício 3 correto.")
taxa_por_grupo(dados, "reviewer_state").iloc[[0, 1, -2, -1]].round(3)

## 6. Relações entre duas variáveis

### Duas variáveis categóricas: mapa de calor

Uma tabela cruzada com muitas células é difícil de ler em números. O **mapa de calor** pinta cada célula conforme o valor. Para uma grandeza que vai de pouco a muito, use uma escala **sequencial**: uma única cor, do claro ao escuro.

In [ ]:
cruzada = pd.crosstab(dados["faixa_etaria"], dados["overall_rating"], normalize="index")      # cada linha soma 1

fig, ax = plt.subplots(figsize=(5.5, 3))
imagem = ax.imshow(cruzada, cmap="Blues", vmin=0, aspect="auto")
ax.set_xticks(range(5), labels=cruzada.columns)
ax.set_yticks(range(len(cruzada)), labels=cruzada.index)
for i in range(cruzada.shape[0]):
    for j in range(cruzada.shape[1]):
        valor = cruzada.iloc[i, j]
        ax.text(j, i, f"{valor:.0%}", ha="center", va="center", fontsize=8, color="white" if valor > 0.25 else "black")
ax.set_xlabel("Nota")
ax.set_ylabel("Faixa etária")
ax.set_title("Os mais jovens dão mais nota 5; os mais velhos, mais nota 4")
ax.grid(False)
fig.colorbar(imagem, ax=ax, format=matplotlib.ticker.PercentFormatter(1.0), label="Proporção na faixa")
plt.show()

### Quando o valor tem um ponto neutro: escala divergente

Se a pergunta é "acima ou abaixo do normal?", o zero da diferença é o ponto neutro. Usa-se uma escala **divergente**: duas cores opostas, com o neutro no meio, **centrada no zero**.

In [ ]:
top_categorias = dados["site_category_lv1"].value_counts().head(8).index
mensal = (
    dados[dados["site_category_lv1"].isin(top_categorias)]
    .assign(mes=lambda t: t["submission_date"].dt.month)
    .pivot_table(index="site_category_lv1", columns="mes", values="nao_recomenda", aggfunc="mean")
)
desvio = mensal - TAXA_GERAL                         # diferença para a taxa geral, em pontos
limite = np.abs(desvio.to_numpy()).max()

fig, ax = plt.subplots(figsize=(6.5, 3.6))
imagem = ax.imshow(desvio, cmap="RdBu_r", vmin=-limite, vmax=limite, aspect="auto")       # vmin e vmax simétricos centram no zero
ax.set_xticks(range(5), labels=["jan", "fev", "mar", "abr", "mai"])
ax.set_yticks(range(len(desvio)), labels=desvio.index)
ax.set_title("Diferença para a taxa geral de não recomendação")
ax.grid(False)
fig.colorbar(imagem, ax=ax, format=matplotlib.ticker.PercentFormatter(1.0, decimals=0), label="acima (vermelho) ou abaixo (azul)")
plt.show()

| A cor representa... | Escala | Exemplos no Matplotlib |
|---|---|---|
| **categorias** sem ordem | cores distintas, poucas (até 6 a 8) | as constantes `AZUL`, `LARANJA`, `VERDE` |
| **quantidade**, de pouco a muito | sequencial: uma cor, do claro ao escuro | `"Blues"`, `"viridis"` |
| **desvio** em torno de um neutro | divergente: duas cores e um meio neutro | `"RdBu_r"`, com `vmin = -vmax` |

> ⚠️ Evite a escala arco-íris (`"jet"`, `"rainbow"`): ela cria fronteiras visuais onde não há nada nos dados e é ilegível para quem tem daltonismo, condição que afeta cerca de 1 em cada 12 homens. Pelo mesmo motivo, nunca dependa só de vermelho contra verde.

### Duas variáveis numéricas: dispersão e sobreposição

Com dezenas de milhares de pontos, o gráfico de dispersão vira um borrão. Duas saídas: **transparência** ou **contagem por região**.

In [ ]:
amostra = dados.dropna(subset=["idade"]).sample(20_000, random_state=0)

fig, (esq, dir_) = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
esq.scatter(amostra["idade"], amostra["palavras"], s=4, alpha=0.15, color=AZUL)
esq.set_title("Dispersão com transparência")
mapa = dir_.hexbin(amostra["idade"], amostra["palavras"], gridsize=30, cmap="Blues", bins="log", extent=(14, 90, 0, 150))
dir_.set_title("Contagem por hexágono")
fig.colorbar(mapa, ax=dir_, label="Avaliações (escala logarítmica)")
for eixo in (esq, dir_):
    eixo.set_xlabel("Idade (anos)")
    eixo.set_ylim(0, 150)
    eixo.grid(False)
esq.set_ylabel("Palavras na avaliação")
plt.show()

### ✏️ Exercício 4 (★★): quando as avaliações negativas chegam?

1. Construa `tabela_ex4`: a taxa de não recomendação por **dia da semana** (linhas, de segunda a domingo) e **período do dia** (colunas `"madrugada"`, `"manhã"`, `"tarde"`, `"noite"`). As células abaixo já criam as duas colunas auxiliares.
2. Desenhe `tabela_ex4` como mapa de calor em um `Axes` chamado `ax4`, com escala sequencial, os rótulos corretos nos dois eixos e uma barra de cores.

In [ ]:
auxiliar = dados.assign(
    dia_semana=pd.Categorical(
        dados["submission_date"].dt.dayofweek.map(dict(enumerate(["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]))),
        categories=["seg", "ter", "qua", "qui", "sex", "sáb", "dom"], ordered=True,
    ),
    periodo=pd.cut(dados["hora"], bins=[-1, 5, 11, 17, 23], labels=["madrugada", "manhã", "tarde", "noite"]),
)

In [ ]:
# ✏️ Exercício 4
# SEU CÓDIGO AQUI
tabela_ex4 = ...
fig, ax4 = plt.subplots(figsize=(5.5, 3.6))
...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
assert tabela_ex4.shape == (7, 4) and list(tabela_ex4.index) == ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
assert tabela_ex4.to_numpy().min() > 0.15 and tabela_ex4.to_numpy().max() < 0.45, "as células devem ser taxas (médias de nao_recomenda)"
assert len(ax4.images) == 1 and [t.get_text() for t in ax4.get_yticklabels()][0] == "seg", "use ax4.imshow e rotule as linhas com os dias"
print("Exercício 4 correto. Amplitude das taxas:", f"{tabela_ex4.to_numpy().min():.1%} a {tabela_ex4.to_numpy().max():.1%}")

## 7. Evolução no tempo

Séries diárias são ruidosas. Uma **média móvel** suaviza o ruído e deixa a tendência visível; mostrar as duas juntas é mais honesto do que mostrar só a suavizada.

In [ ]:
diario = dados.groupby("dia").agg(avaliacoes=("nao_recomenda", "size"), taxa=("nao_recomenda", "mean"))
diario["taxa_7d"] = diario["taxa"].rolling(7, center=True).mean()

fig, (cima, baixo) = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True, height_ratios=[1, 1.4])

cima.fill_between(diario.index, diario["avaliacoes"], color=CINZA, alpha=0.35)
cima.set_ylabel("Avaliações por dia")
cima.set_title("Em fevereiro o volume cai e a insatisfação dispara")

baixo.plot(diario.index, diario["taxa"], color=AZUL, alpha=0.3, linewidth=1, label="diária")
baixo.plot(diario.index, diario["taxa_7d"], color=AZUL, linewidth=2, label="média móvel de 7 dias")
baixo.set_ylabel("Não recomendam")
baixo.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
baixo.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()

> ⚠️ **Nunca use dois eixos verticais no mesmo gráfico** para colocar duas medidas de escalas diferentes (volume e taxa, por exemplo). A posição em que as curvas se cruzam depende apenas de como cada escala foi esticada, e o leitor enxerga relações que não existem. A solução é a que usamos: **dois painéis** empilhados, com o eixo do tempo compartilhado.

### ✏️ Exercício 5 (★★): série semanal

Construa `serie_ex5`, uma tabela indexada por semana com as colunas `taxa` (taxa de não recomendação na semana) e `movel` (média móvel de 4 semanas de `taxa`). Depois desenhe as duas em um `Axes` chamado `ax5`, com legenda.

Dica: `dados.set_index("submission_date").resample("W")`.

In [ ]:
# ✏️ Exercício 5
# SEU CÓDIGO AQUI
serie_ex5 = ...
fig, ax5 = plt.subplots()
...

In [ ]:
# ✅ Verificação do exercício 5 (não altere)
assert list(serie_ex5.columns) == ["taxa", "movel"] and 20 <= len(serie_ex5) <= 23
assert serie_ex5["movel"].isna().sum() == 3, "a média móvel de 4 semanas só existe a partir da quarta semana"
assert np.isclose(serie_ex5["movel"].iloc[3], serie_ex5["taxa"].iloc[:4].mean())
assert len(ax5.lines) >= 2 and ax5.get_legend() is not None, "desenhe as duas séries e inclua a legenda"
print("Exercício 5 correto.")

## 8. seaborn: estatística com menos código

O **seaborn** é construído sobre o Matplotlib e parte de uma tabela: você diz quais colunas vão em quais papéis (`x`, `y`, `hue`) e ele agrega, calcula intervalos e desenha. Cada função devolve um `Axes`, que continua ajustável com tudo o que você já sabe.

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(6, 3.2))
sns.barplot(data=dados, x="faixa_etaria", y="nao_recomenda", hue="reviewer_gender", palette=[AZUL, LARANJA],
            errorbar=("ci", 95), n_boot=200, seed=0, ax=ax)          # calcula a média e o intervalo de cada grupo
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_xlabel("Faixa etária")
ax.set_ylabel("Não recomendam")
ax.set_title("A insatisfação é maior entre os 25 e os 59 anos")
ax.legend(title="Gênero", frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

Com `col=`, as funções de figura (`catplot`, `relplot`, `displot`) criam **pequenos múltiplos**: o mesmo gráfico repetido para cada valor de uma variável, com eixos comparáveis.

In [ ]:
quatro = dados[dados["site_category_lv1"].isin(dados["site_category_lv1"].value_counts().head(4).index)]

grade = sns.displot(data=quatro, x="overall_rating", col="site_category_lv1", col_wrap=4, discrete=True, stat="proportion",
                    common_norm=False, height=2.4, aspect=1.05, color=AZUL, shrink=0.8)
grade.set_titles("{col_name}")
grade.set_axis_labels("Nota", "Proporção")
plt.show()

## 9. Plotly: gráficos interativos

Em exploração, poder passar o cursor e ler o valor exato economiza tempo. O **Plotly Express** produz gráficos interativos com uma chamada.

In [ ]:
import plotly.express as px

figura = px.scatter(
    por_sub.reset_index().query("n >= 300"),
    x="n", y="taxa", hover_name="site_category_lv2", log_x=True,
    labels={"n": "Avaliações (escala logarítmica)", "taxa": "Taxa de não recomendação"},
    title="Passe o cursor: cada ponto é uma subcategoria",
)
figura.update_traces(marker={"color": AZUL, "size": 8, "opacity": 0.7})
figura.add_hline(y=TAXA_GERAL, line_dash="dash", line_color=CINZA)
figura.update_layout(height=380, template="simple_white", yaxis_tickformat=".0%")
figura.show()

| Biblioteca | Use quando |
|---|---|
| **Matplotlib** | precisa de controle total; figura para relatório ou artigo |
| **seaborn** | quer agregação, intervalos e pequenos múltiplos a partir de uma tabela |
| **Plotly** | está explorando e quer interatividade; vai publicar em página ou painel |
| **`.plot` do pandas** | quer só espiar uma série durante a análise |

## 10. Gráficos que enganam

O gráfico abaixo foi feito com os mesmos dados de região do notebook 02. Tudo nele está "certo" e a impressão que ele passa está errada.

In [ ]:
regioes = {"Norte": ["AC", "AP", "AM", "PA", "RO", "RR", "TO"], "Nordeste": ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"],
           "Centro-Oeste": ["DF", "GO", "MT", "MS"], "Sudeste": ["ES", "MG", "RJ", "SP"], "Sul": ["PR", "RS", "SC"]}
uf_para_regiao = {uf: regiao for regiao, ufs in regioes.items() for uf in ufs}
por_regiao = dados.assign(regiao=dados["reviewer_state"].map(uf_para_regiao)).groupby("regiao")["nao_recomenda"].agg(["mean", "size"])

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.bar(por_regiao.index, por_regiao["mean"], color=["red", "gold", "limegreen", "deepskyblue", "magenta"])
ax.set_ylim(0.26, 0.285)                              # ⚠️ eixo cortado
ax.set_title("Taxa por região")
plt.show()

### 🔮 Antes de continuar

Liste tudo o que há de errado nesse gráfico. Tente chegar a pelo menos quatro itens.

✏️ **Sua resposta:**

_Escreva aqui._

### ✏️ Exercício 6 (★★★): o gráfico honesto

Escreva `grafico_honesto(ax)`, que desenha no `Axes` recebido a taxa de não recomendação por região, usando `por_regiao`, de forma correta:

- barras partindo do **zero**, ordenadas pelo valor, todas da **mesma cor**;
- o valor escrito junto a cada barra;
- eixo de valor com rótulo, e um título que diga a conclusão.

In [ ]:
# ✏️ Exercício 6
def grafico_honesto(ax):
    # SEU CÓDIGO AQUI
    ...

fig, ax6 = plt.subplots(figsize=(5.5, 3))
grafico_honesto(ax6)
plt.show()

In [ ]:
# ✅ Verificação do exercício 6 (não altere)
assert len(ax6.patches) == 5, "uma barra por região"
_horizontal = len({round(p.get_height(), 6) for p in ax6.patches}) == 1      # em barras horizontais, a espessura é igual
_tamanhos = [p.get_width() if _horizontal else p.get_height() for p in ax6.patches]
_inicio = (ax6.get_xlim() if _horizontal else ax6.get_ylim())[0]
assert _inicio == 0, "o eixo de valor deve começar em zero"
assert _tamanhos == sorted(_tamanhos) or _tamanhos == sorted(_tamanhos, reverse=True), "ordene as barras pelo valor"
assert len({p.get_facecolor() for p in ax6.patches}) == 1, "todas as barras medem a mesma coisa: use uma única cor"
assert len(ax6.texts) >= 5 and titulo_de(ax6) and (ax6.get_xlabel() or ax6.get_ylabel()), "faltam os valores nas barras, o título ou o rótulo do eixo"
print("Exercício 6 correto.")

## 11. Um roteiro de análise exploratória

Explorar sem roteiro vira uma coleção de gráficos. Quando há um **alvo** (aqui, `nao_recomenda`), a análise exploratória tem uma finalidade: decidir o que entra no modelo e que cuidados tomar.

| Passo | Pergunta | Onde fizemos |
|---|---|---|
| 1. Estrutura | Quantas linhas, que tipos, quantos valores distintos? | notebook 02 |
| 2. Qualidade | Onde faltam dados? Há valores impossíveis? | notebook 02 |
| 3. O alvo | Qual é a proporção de cada classe? | 27% não recomendam |
| 4. Uma variável por vez | Como cada variável se distribui? | seção 4 |
| 5. Cada variável contra o alvo | Quais mostram relação? Com que incerteza? | seções 5 e 6, e abaixo |
| 6. O tempo | O alvo muda ao longo do período? | seção 7 |
| 7. A estrutura | Há linhas dependentes (mesmo cliente, mesmo produto)? | notebook 02 |
| 8. Síntese | O que levar para o modelo? | abaixo |

O passo 5 pode ser feito de uma vez, com a função do exercício 3 em pequenos múltiplos. (A célula abaixo traz a sua própria versão da função, para não depender do exercício.)

In [ ]:
def resumo_alvo(tabela, coluna, minimo=300):
    return (
        tabela.groupby(coluna, observed=True).agg(n=("nao_recomenda", "size"), taxa=("nao_recomenda", "mean"))
        .query("n >= @minimo").assign(margem=lambda t: 1.96 * np.sqrt(t["taxa"] * (1 - t["taxa"]) / t["n"]))
    )

exploracao = auxiliar.assign(
    texto_longo=pd.cut(auxiliar["palavras"], bins=[0, 10, 20, 40, 5000], labels=["até 10", "11 a 20", "21 a 40", "mais de 40"]),
    cita_entrega=np.where(auxiliar["texto"].str.contains("entrega|prazo|atras", case=False, regex=True), "sim", "não"),
)
paineis = [("faixa_etaria", "Faixa etária"), ("reviewer_gender", "Gênero"), ("periodo", "Período do dia"),
           ("dia_semana", "Dia da semana"), ("texto_longo", "Palavras no texto"), ("cita_entrega", "Cita entrega ou prazo")]

fig, eixos = plt.subplots(2, 3, figsize=(11, 5.4), sharey=True)
for eixo, (coluna, titulo) in zip(eixos.ravel(), paineis):
    r = resumo_alvo(exploracao, coluna)
    eixo.errorbar(range(len(r)), r["taxa"], yerr=r["margem"], fmt="o", color=AZUL, ecolor=CINZA, capsize=3)
    eixo.set_xticks(range(len(r)), labels=r.index, fontsize=8)
    eixo.set_xlim(-0.6, len(r) - 0.4)
    eixo.axhline(TAXA_GERAL, color=CINZA, linestyle="--", linewidth=1)
    eixo.set_title(titulo)
    eixo.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
eixos[0, 0].set_ylabel("Não recomendam")
eixos[1, 0].set_ylabel("Não recomendam")
fig.suptitle("O tamanho do texto diz muito mais que o perfil do cliente", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
plt.show()

Todos os painéis têm a **mesma escala vertical**, e isso é o que permite a comparação entre eles. As variáveis de perfil e de horário oscilam poucos pontos em torno da linha tracejada. O tamanho do texto, sozinho, leva a taxa de 17% a mais de 50%.

O último painel traz uma surpresa: as avaliações que citam entrega ou prazo têm **menos** insatisfação que as demais. Muita gente escreve para elogiar a entrega rápida. É um bom lembrete de que hipóteses plausíveis ("quem fala de entrega está reclamando") precisam ser conferidas nos dados antes de virarem variável do modelo.

### Síntese: o que levamos para o modelo

1. **O alvo é desbalanceado** (27% de positivos): acurácia não servirá como métrica.
2. **Perfil e horário têm sinal fraco**: idade, gênero, estado, dia e hora movem a taxa em poucos pontos.
3. **O produto importa**: categorias e subcategorias diferem bastante, mas as subcategorias pequenas têm taxas instáveis e pedem agrupamento.
4. **O texto concentra o sinal**: só o tamanho já separa bem os grupos, e o conteúdo precisa ser lido com cuidado (citar a entrega não significa reclamar dela).
5. **Há dependência entre linhas**: clientes e produtos repetidos, e mudança ao longo do tempo, pedem cuidado na divisão em treino e teste.

O notebook 05 transforma cada um desses itens em uma decisão de modelagem.

### ✏️ Exercício 7 (★★★): sua investigação

Escolha **uma** pergunta sobre estes dados que ainda não foi respondida neste notebook (por exemplo: a marca informada ou ausente faz diferença? Títulos em maiúsculas indicam insatisfação? Avaliações com ponto de exclamação são mais positivas?). Responda com **um gráfico**, em um `Axes` chamado `ax7`, que mostre a incerteza quando houver comparação entre grupos. Depois escreva a conclusão em duas ou três frases.

In [ ]:
# ✏️ Exercício 7
# SEU CÓDIGO AQUI
fig, ax7 = plt.subplots()
...

In [ ]:
# ✅ Verificação do exercício 7 (não altere)
assert titulo_de(ax7) and (ax7.get_xlabel() or ax7.get_ylabel()), "o gráfico precisa de título e de rótulo no eixo de valor"
assert len(ax7.lines) + len(ax7.patches) + len(ax7.collections) + len(ax7.images) > 0, "o Axes está vazio"
print("Estrutura mínima conferida. A qualidade da resposta está na conclusão escrita.")

**Sua conclusão** (pergunta, o que o gráfico mostra, e o que isso sugere para o modelo):

✏️ **Sua resposta:**

_Escreva aqui._

## 12. Para levar

### Erros comuns

| Erro | Como evitar |
|---|---|
| Escolher o gráfico pelo hábito | formular a pergunta primeiro (seção 3) |
| Barras com eixo cortado | eixo de valor começando em zero |
| Categorias em ordem alfabética | ordenar pelo valor, salvo ordem natural |
| Cores sem função, arco-íris, vermelho contra verde | uma cor por papel; sequencial para quantidade; divergente centrada no neutro |
| Dois eixos verticais | dois painéis com o eixo horizontal compartilhado |
| Comparar grupos sem olhar o tamanho deles | barras de erro; filtro de volume mínimo |
| Título com o nome das variáveis | título com a conclusão |
| Dispersão com milhares de pontos sobrepostos | transparência ou contagem por região |

### Resumo

- Resumos numéricos escondem a forma dos dados; olhe antes de modelar.
- No Matplotlib, crie `fig` e `ax` e trabalhe com os métodos do `Axes`.
- A pergunta determina a forma: distribuição, comparação, relação, evolução ou composição.
- Toda taxa por grupo vem com incerteza, e grupos pequenos ocupam os extremos por acaso.
- A cor tem função: identificar categorias, indicar quantidade ou indicar desvio.
- A análise exploratória termina em uma lista de decisões para o modelo.

### O que vem a seguir

No notebook 04, as mesmas perguntas feitas ao pandas serão respondidas com Polars e com SQL no DuckDB, ferramentas que continuam rápidas quando a tabela deixa de caber confortavelmente na memória.

### Para casa

A lista de exercícios está em `03_casa.ipynb`.